1. Introdução

A análise de crédito é uma atividade importante para instituições financeiras, pois permite avaliar o perfil dos clientes e identificar o risco relacionado à concessão de crédito. Com o aumento da quantidade de informações disponíveis, o uso de dados pode auxiliar na criação de análises mais eficientes e apoiar a tomada de decisões.
Neste projeto, desenvolvido para a empresa Data Girls Finance, será realizada uma análise utilizando técnicas de Ciência de Dados e Machine Learning com o objetivo de criar um modelo capaz de classificar o score de crédito dos clientes.
Para isso, será utilizado o dataset 'Credit Score Classification', que contém informações relacionadas ao perfil financeiro dos clientes, como renda, histórico de pagamentos, dívidas, empréstimos, utilização de crédito e outros comportamentos financeiros. A partir desses dados, serão realizadas etapas de exploração, limpeza, análise dos dados, construção de modelos preditivos e avaliação dos resultados. 
Além da previsão do score de crédito, o projeto busca identificar quais características possuem maior relação com o comportamento financeiro dos clientes e gerar insights que possam auxiliar processos de análise de crédito, considerando também a importância da qualidade dos dados e do uso responsável de modelos preditivos.

2. Importação das Bibliotecas

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV

from sklearn.preprocessing import LabelEncoder

from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

from sklearn.ensemble import RandomForestClassifier

from sklearn.tree import DecisionTreeClassifier

from sklearn.linear_model import LogisticRegression

In [2]:
sns.set_style("whitegrid")

3. Leitura dos dados

In [3]:
df = pd.read_csv("../dados/train.csv")

C:\Users\BRUNA\AppData\Local\Temp\ipykernel_8664\261314804.py:1: DtypeWarning: Columns (26) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../dados/train.csv")


In [4]:
df.shape

(100000, 28)

A verificação das dimensões do dataset mostrou que a base é composta por 100.000 registros e 28 variáveis.

In [5]:
df.head()

,ID,Customer_ID,Month,Name,Age,SSN,Occupation,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,...,Credit_Mix,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Payment_Behaviour,Monthly_Balance,Credit_Score
0,0x1602,CUS_0xd40,January,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,1824.843333,3,...,_,809.98,26.822620,22 Years and 1 Months,No,49.574949,80.41529543900253,High_spent_Small_value_payments,312.49408867943663,Good
1,0x1603,CUS_0xd40,February,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,NaN,3,...,Good,809.98,31.944960,NaN,No,49.574949,118.28022162236736,Low_spent_Large_value_payments,284.62916249607184,Good
2,0x1604,CUS_0xd40,March,Aaron Maashoh,-500,821-00-0265,Scientist,19114.12,NaN,3,...,Good,809.98,28.609352,22 Years and 3 Months,No,49.574949,81.699521264648,Low_spent_Medium_value_payments,331.2098628537912,Good
3,0x1605,CUS_0xd40,April,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,NaN,3,...,Good,809.98,31.377862,22 Years and 4 Months,No,49.574949,199.4580743910713,Low_spent_Small_value_payments,223.45130972736786,Good
4,0x1606,CUS_0xd40,May,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,1824.843333,3,...,Good,809.98,24.797347,22 Years and 5 Months,No,49.574949,41.420153086217326,High_spent_Medium_value_payments,341.48923103222177,Good


A visualização inicial do dataset permitiu identificar que os dados são compostos por informações relacionadas ao perfil financeiro dos clientes. Cada registro representa uma observação de um cliente, contendo características pessoais, informações financeiras, histórico de crédito e comportamento de pagamento.
Entre as variáveis disponíveis estão idade, renda anual, quantidade de contas bancárias, dívidas pendentes, histórico de crédito e a classificação final do score de crédito (Credit_Score), que será utilizado como variável alvo (target) no modelo de Machine Learning.

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 28 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   ID                        100000 non-null  object 
 1   Customer_ID               100000 non-null  object 
 2   Month                     100000 non-null  object 
 3   Name                      90015 non-null   object 
 4   Age                       100000 non-null  object 
 5   SSN                       100000 non-null  object 
 6   Occupation                100000 non-null  object 
 7   Annual_Income             100000 non-null  object 
 8   Monthly_Inhand_Salary     84998 non-null   float64
 9   Num_Bank_Accounts         100000 non-null  int64  
 10  Num_Credit_Card           100000 non-null  int64  
 11  Interest_Rate             100000 non-null  int64  
 12  Num_of_Loan               100000 non-null  object 
 13  Type_of_Loan              88592 non-null   ob

A análise da estrutura do dataset permitiu identificar os tipos de dados presentes em cada variável e avaliar sua qualidade para as etapas seguintes do projeto. O conjunto é composto por 20 variáveis do tipo object, 4 do tipo float e 4 do tipo int.
Foi possível observar que algumas variáveis classificadas como object, como idade, renda anual e número de empréstimos, representam informações que deverão ser convertidas para formatos numéricos durante a etapa de preparação dos dados, uma vez que seus valores serão utilizados na modelagem preditiva.
Também foram identificados valores ausentes em colunas como 'Name', 'Monthly_Inhand_Salary', 'Type_of_Loan', 'Num_of_Delayed_Payment' e 'Num_Credit_Inquiries'. Essas ocorrências serão investigadas para definir a estratégia de tratamento mais adequada, buscando preservar a qualidade das informações.
Além disso, observou-se que a variável 'Credit_Score' corresponde à variável alvo do modelo de Machine Learning, enquanto as demais colunas representam características dos clientes que poderão contribuir para a classificação do score de crédito.

Observação: Durante a importação do dataset, foi exibido um alerta (DtypeWarning) indicando a presença de tipos de dados mistos em uma das colunas. Essa inconsistência será investigada e tratada posteriormente na etapa de limpeza e preparação dos dados.

In [7]:
df.describe()

,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Delay_from_due_date,Num_Credit_Inquiries,Credit_Utilization_Ratio,Total_EMI_per_month
count,84998.000000,100000.000000,100000.00000,100000.000000,100000.000000,98035.000000,100000.000000,100000.000000
mean,4194.170850,17.091280,22.47443,72.466040,21.068780,27.754251,32.285173,1403.118217
std,3183.686167,117.404834,129.05741,466.422621,14.860104,193.177339,5.116875,8306.041270
min,303.645417,-1.000000,0.00000,1.000000,-5.000000,0.000000,20.000000,0.000000
25%,1625.568229,3.000000,4.00000,8.000000,10.000000,3.000000,28.052567,30.306660
50%,3093.745000,6.000000,5.00000,13.000000,18.000000,6.000000,32.305784,69.249473
75%,5957.448333,7.000000,7.00000,20.000000,28.000000,9.000000,36.496663,161.224249
max,15204.633333,1798.000000,1499.00000,5797.000000,67.000000,2597.000000,50.000000,82331.000000


A análise estatística inicial das variáveis numéricas permitiu observar a distribuição dos dados, identificando medidas de tendência central, dispersão e possíveis inconsistências que deverão ser tratadas antes da aplicação dos modelos de Machine Learning.
Algumas variáveis apresentaram comportamento consistente, como 'Credit_Utilization_Ratio', que possui baixa dispersão em relação à média e valores distribuídos dentro de uma faixa esperada para a característica analisada. A variável 'Delay_from_due_date' também apresentou uma distribuição central coerente, embora tenha sido identificado um valor mínimo negativo que deverá ser investigado.
Por outro lado, algumas variáveis apresentaram grande diferença entre média, mediana e valores máximos, indicando possível influência de valores extremos. Foram observadas possíveis inconsistências em variáveis como Num_Bank_Accounts, que apresentou valores incompatíveis como -1 e 1798, Num_Credit_Card, com valores extremos de até 1499 cartões, 'Interest_Rate', com valores muito acima da distribuição principal, 'Num_Credit_Inquiries' e 'Total_EMI_per_month', que apresentaram altos desvios padrão e possíveis outliers.
Também foram identificados valores ausentes em algumas variáveis, como 'Monthly_Inhand_Salary' e 'Num_Credit_Inquiries', que serão investigados e tratados na etapa de limpeza dos dados.
De forma geral, a análise indica que algumas variáveis possuem grande potencial para explicar o comportamento financeiro dos clientes e auxiliar na classificação do score de crédito. Variáveis relacionadas à renda, histórico de pagamentos, utilização de crédito, endividamento e comprometimento financeiro serão analisadas posteriormente para identificar sua relação com a variável alvo 'Credit_Score'.

4. Exploração inicial

In [8]:
df.isnull().sum()

ID                              0
Customer_ID                     0
Month                           0
Name                         9985
Age                             0
SSN                             0
Occupation                      0
Annual_Income                   0
Monthly_Inhand_Salary       15002
Num_Bank_Accounts               0
Num_Credit_Card                 0
Interest_Rate                   0
Num_of_Loan                     0
Type_of_Loan                11408
Delay_from_due_date             0
Num_of_Delayed_Payment       7002
Changed_Credit_Limit            0
Num_Credit_Inquiries         1965
Credit_Mix                      0
Outstanding_Debt                0
Credit_Utilization_Ratio        0
Credit_History_Age           9030
Payment_of_Min_Amount           0
Total_EMI_per_month             0
Amount_invested_monthly      4479
Payment_Behaviour               0
Monthly_Balance              1200
Credit_Score                    0
dtype: int64

A análise dos valores ausentes mostrou que a maior parte das variáveis do dataset está completa, não apresentando registros faltantes. Apenas algumas colunas possuem valores ausentes, sendo as principais: 'Monthly_Inhand_Salary', 'Type_of_Loan', 'Name', 'Credit_History_Age', Num_of_Delayed_Payment', 'Amount_invested_monthly', 'Num_Credit_Inquiries' e 'Monthly_Balance'.
Embora algumas dessas colunas apresentem milhares de valores ausentes em números absolutos, essa quantidade representa uma parcela relativamente pequena em relação ao total de 100.000 registros do dataset.


In [9]:
(df.isnull().sum() / len(df) * 100).sort_values(ascending=False)

Monthly_Inhand_Salary       15.002
Type_of_Loan                11.408
Name                         9.985
Credit_History_Age           9.030
Num_of_Delayed_Payment       7.002
Amount_invested_monthly      4.479
Num_Credit_Inquiries         1.965
Monthly_Balance              1.200
Month                        0.000
Age                          0.000
Customer_ID                  0.000
ID                           0.000
Interest_Rate                0.000
Num_Credit_Card              0.000
Num_Bank_Accounts            0.000
Annual_Income                0.000
SSN                          0.000
Occupation                   0.000
Delay_from_due_date          0.000
Num_of_Loan                  0.000
Outstanding_Debt             0.000
Credit_Mix                   0.000
Changed_Credit_Limit         0.000
Credit_Utilization_Ratio     0.000
Total_EMI_per_month          0.000
Payment_of_Min_Amount        0.000
Payment_Behaviour            0.000
Credit_Score                 0.000
dtype: float64

A análise percentual dos valores ausentes mostrou que nenhuma variável apresenta mais de 15% de registros faltantes. 'Monthly_Inhand_Salary' concentra o maior percentual de ausências (15,0%), seguida por 'Type_of_Loan' (11,4%) e 'Name' (10,0%). As demais variáveis possuem menos de 10% de valores ausentes, indicando que o conjunto de dados apresenta boa completude para o desenvolvimento das análises e dos modelos preditivos.

In [10]:
df.duplicated().sum()

np.int64(0)

A verificação de registros duplicados mostrou que o dataset não possui linhas duplicadas. Isso indica que cada registro representa uma observação distinta, não sendo necessária a remoção de duplicatas nesta etapa da análise.

In [11]:
df["Credit_Score"].value_counts()

Credit_Score
Standard    53174
Poor        28998
Good        17828
Name: count, dtype: int64

A distribuição da variável alvo ('Credit_Score') mostra que a classe 'Standard' é a mais frequente no conjunto de dados, representando aproximadamente 53,2% dos registros. Em seguida, aparecem as classes 'Poor' (29,0%) e 'Good' (17,8%).
Embora exista um desequilíbrio entre as classes, todas possuem uma quantidade expressiva de registros, indicando que o dataset apresenta uma distribuição adequada para o treinamento de modelos de classificação. Além disso, a variável alvo não apresenta valores ausentes, garantindo que todos os registros possam ser utilizados durante o treinamento e a avaliação dos modelos preditivos.

5. Limpeza

Nessa fase, foram tratados problemas que poderiam comprometer a qualidade das análises e o desempenho do modelo de Machine Learning, como espaços em branco, valores inválidos, tipos de dados incorretos e valores ausentes. O objetivo é garantir um conjunto de dados consistente e adequado para as próximas etapas do projeto.

5.1. Remoção de espaços em branco

In [12]:
df.columns = df.columns.str.strip()

Para garantir a padronização dos nomes das colunas e evitar problemas durante a manipulação do dataset, foi realizada a remoção de espaços em branco no início e no final dos nomes das variáveis.

5.2. Tratamento de valores inválidos

Nessa etapa foi realizada uma análise individual das variáveis categóricas para identificação de valores inválidos ou utilizados como representação de dados ausentes. Foram encontrados diferentes padrões de valores inválidos em algumas colunas, como caracteres especiais e marcadores de ausência de informação.
Cada ocorrência identificada foi avaliada individualmente e substituída por 'NaN' quando representava ausência de informação ou não correspondia a uma categoria válida de variável. Esse procedimento irá nos permitir padronizar os dados e facilitar o tratamento dos valores ausentes nas etapas seguintes.

In [13]:
df.select_dtypes(include="object").columns

Index(['ID', 'Customer_ID', 'Month', 'Name', 'Age', 'SSN', 'Occupation',
       'Annual_Income', 'Num_of_Loan', 'Type_of_Loan',
       'Num_of_Delayed_Payment', 'Changed_Credit_Limit', 'Credit_Mix',
       'Outstanding_Debt', 'Credit_History_Age', 'Payment_of_Min_Amount',
       'Amount_invested_monthly', 'Payment_Behaviour', 'Monthly_Balance',
       'Credit_Score'],
      dtype='object')

In [14]:
df["SSN"].unique()

array(['821-00-0265', '#F%$D@*&8', '004-07-5839', ..., '133-16-7738',
       '031-35-0942', '078-73-5990'], shape=(12501,), dtype=object)

In [15]:
df["SSN"] = df["SSN"].replace("#F%$D@*&8", np.nan)

In [16]:
df["SSN"].value_counts(dropna=False)

SSN
NaN            5572
864-24-3672       8
094-81-5856       8
647-67-8889       8
425-47-6723       8
               ... 
604-62-6133       4
331-28-1921       4
838-33-4811       4
856-06-6147       4
753-72-2651       4
Name: count, Length: 12501, dtype: int64

In [17]:
df["Occupation"].unique()

array(['Scientist', '_______', 'Teacher', 'Engineer', 'Entrepreneur',
       'Developer', 'Lawyer', 'Media_Manager', 'Doctor', 'Journalist',
       'Manager', 'Accountant', 'Musician', 'Mechanic', 'Writer',
       'Architect'], dtype=object)

In [18]:
df["Occupation"] = df["Occupation"].replace("_______", np.nan)

In [19]:
df["Occupation"].value_counts(dropna=False)

Occupation
NaN              7062
Lawyer           6575
Architect        6355
Engineer         6350
Scientist        6299
Mechanic         6291
Accountant       6271
Developer        6235
Media_Manager    6232
Teacher          6215
Entrepreneur     6174
Doctor           6087
Journalist       6085
Manager          5973
Musician         5911
Writer           5885
Name: count, dtype: int64

In [20]:
df["Changed_Credit_Limit"].unique()

array(['11.27', '_', '6.27', ..., '17.509999999999998', '25.16', '21.17'],
      shape=(4384,), dtype=object)

In [21]:
df["Changed_Credit_Limit"] = df["Changed_Credit_Limit"].replace("_", np.nan)

In [22]:
df["Changed_Credit_Limit"].value_counts(dropna=False)

Changed_Credit_Limit
NaN                  2091
8.22                  133
11.5                  127
11.32                 126
7.35                  121
                     ... 
30.16                   1
4.710000000000001       1
-4.39                   1
27.38                   1
16.63                   1
Name: count, Length: 4384, dtype: int64

In [23]:
df["Credit_Mix"].unique()

array(['_', 'Good', 'Standard', 'Bad'], dtype=object)

In [24]:
df["Credit_Mix"] = df["Credit_Mix"].replace("_", np.nan)

In [25]:
df["Credit_Mix"].value_counts(dropna=False)

Credit_Mix
Standard    36479
Good        24337
NaN         20195
Bad         18989
Name: count, dtype: int64

In [26]:
df["Payment_of_Min_Amount"].unique()

array(['No', 'NM', 'Yes'], dtype=object)

In [27]:
df["Payment_of_Min_Amount"] = df["Payment_of_Min_Amount"].replace("NM", np.nan)

In [28]:
df["Payment_of_Min_Amount"].value_counts(dropna=False)

Payment_of_Min_Amount
Yes    52326
No     35667
NaN    12007
Name: count, dtype: int64

In [29]:
df["Payment_Behaviour"].unique()

array(['High_spent_Small_value_payments',
       'Low_spent_Large_value_payments',
       'Low_spent_Medium_value_payments',
       'Low_spent_Small_value_payments',
       'High_spent_Medium_value_payments', '!@9#%8',
       'High_spent_Large_value_payments'], dtype=object)

In [30]:
df["Payment_Behaviour"] = df["Payment_Behaviour"].replace("!@9#%8", np.nan)

In [31]:
df["Payment_Behaviour"].value_counts(dropna=False)

Payment_Behaviour
Low_spent_Small_value_payments      25513
High_spent_Medium_value_payments    17540
Low_spent_Medium_value_payments     13861
High_spent_Large_value_payments     13721
High_spent_Small_value_payments     11340
Low_spent_Large_value_payments      10425
NaN                                  7600
Name: count, dtype: int64

5.3. Conversão dos tipos de dados

Nessa etapa, foi realizada a verificação dos tipos de dados das colunas para identificar variáveis armazenadas em formatos inadequados.
Algumas colunas numéricas estavam classificadas como 'object' devido à presença de valores representados como texto ou caracteres inconsistentes. Após a limpeza desses registros, foi realizada a conversão para os tipos numéricos adequados ('int' e 'float'), garantindo que os dados fossem interpretados corretamente nas etapas de análise e modelagem.
A coluna 'Credit_History_Age' foi mantida temporariamente como 'object', pois apresenta informações de tempo no formato textual, exigindo uma transformação específica antes de sua utilização no modelo.

In [32]:
df.dtypes

ID                           object
Customer_ID                  object
Month                        object
Name                         object
Age                          object
SSN                          object
Occupation                   object
Annual_Income                object
Monthly_Inhand_Salary       float64
Num_Bank_Accounts             int64
Num_Credit_Card               int64
Interest_Rate                 int64
Num_of_Loan                  object
Type_of_Loan                 object
Delay_from_due_date           int64
Num_of_Delayed_Payment       object
Changed_Credit_Limit         object
Num_Credit_Inquiries        float64
Credit_Mix                   object
Outstanding_Debt             object
Credit_Utilization_Ratio    float64
Credit_History_Age           object
Payment_of_Min_Amount        object
Total_EMI_per_month         float64
Amount_invested_monthly      object
Payment_Behaviour            object
Monthly_Balance              object
Credit_Score                

In [33]:
df["Age"].unique()

array(['23', '-500', '28_', ..., '4808_', '2263', '1342'],
      shape=(1788,), dtype=object)

In [34]:
df["Age"] = (
    df["Age"]
    .str.replace("_", "", regex=False)
)

In [35]:
df["Age"] = pd.to_numeric(
    df["Age"],
    errors="coerce"
)

In [36]:
df["Annual_Income"].unique()

array(['19114.12', '34847.84', '34847.84_', ..., '20002.88', '39628.99',
       '39628.99_'], shape=(18940,), dtype=object)

In [37]:
df["Annual_Income"] = (
    df["Annual_Income"]
    .str.replace("_", "", regex=False)
)

In [38]:
df["Annual_Income"] = pd.to_numeric(
    df["Annual_Income"],
    errors="coerce"
)

In [39]:
df["Num_of_Loan"].unique()

array(['4', '1', '3', '967', '-100', '0', '0_', '2', '3_', '2_', '7', '5',
       '5_', '6', '8', '8_', '9', '9_', '4_', '7_', '1_', '1464', '6_',
       '622', '352', '472', '1017', '945', '146', '563', '341', '444',
       '720', '1485', '49', '737', '1106', '466', '728', '313', '843',
       '597_', '617', '119', '663', '640', '92_', '1019', '501', '1302',
       '39', '716', '848', '931', '1214', '186', '424', '1001', '1110',
       '1152', '457', '1433', '1187', '52', '1480', '1047', '1035',
       '1347_', '33', '193', '699', '329', '1451', '484', '132', '649',
       '995', '545', '684', '1135', '1094', '1204', '654', '58', '348',
       '614', '1363', '323', '1406', '1348', '430', '153', '1461', '905',
       '1312', '1424', '1154', '95', '1353', '1228', '819', '1006', '795',
       '359', '1209', '590', '696', '1185_', '1465', '911', '1181', '70',
       '816', '1369', '143', '1416', '455', '55', '1096', '1474', '420',
       '1131', '904', '89', '1259', '527', '1241', '449', 

In [40]:
df["Num_of_Loan"] = (
    df["Num_of_Loan"]
    .str.replace("_", "", regex=False)
)

In [41]:
df["Num_of_Loan"] = pd.to_numeric(
    df["Num_of_Loan"],
    errors="coerce"
)

In [42]:
df["Num_of_Delayed_Payment"].unique()

array(['7', nan, '4', '8_', '6', '1', '-1', '3_', '0', '8', '5', '3', '9',
       '12', '15', '17', '10', '2', '2_', '11', '14', '20', '22', '13',
       '13_', '14_', '16', '12_', '18', '19', '23', '24', '21', '3318',
       '3083', '22_', '1338', '4_', '26', '11_', '3104', '21_', '25',
       '10_', '183_', '9_', '1106', '834', '19_', '24_', '17_', '23_',
       '2672', '20_', '2008', '-3', '538', '6_', '1_', '16_', '27', '-2',
       '3478', '2420', '15_', '707', '708', '26_', '18_', '3815', '28',
       '5_', '1867', '2250', '1463', '25_', '7_', '4126', '2882', '1941',
       '2655', '2628', '132', '3069', '306', '0_', '3539', '3684', '1823',
       '4128', '1946', '827', '2297', '2566', '904', '182', '929', '3568',
       '2503', '1552', '2812', '1697', '3764', '851', '3905', '923', '88',
       '1668', '3253', '808', '2689', '3858', '642', '3457', '1402',
       '1732', '3154', '847', '3037', '2204', '3103', '1063', '2056',
       '1282', '1841', '2569_', '211', '793', '3484', '4

In [43]:
df["Num_of_Delayed_Payment"] = (
    df["Num_of_Delayed_Payment"]
    .str.replace("_", "", regex=False)
)

In [44]:
df["Num_of_Delayed_Payment"] = pd.to_numeric(
    df["Num_of_Delayed_Payment"],
    errors="coerce"
)

In [45]:
df["Changed_Credit_Limit"].unique()

array(['11.27', nan, '6.27', ..., '17.509999999999998', '25.16', '21.17'],
      shape=(4384,), dtype=object)

In [46]:
df["Changed_Credit_Limit"] = pd.to_numeric(
    df["Changed_Credit_Limit"],
    errors="coerce"
)

In [47]:
df["Outstanding_Debt"].unique()

array(['809.98', '605.03', '1303.01', ..., '3571.7_', '3571.7', '502.38'],
      shape=(13178,), dtype=object)

In [48]:
df["Outstanding_Debt"] = (
    df["Outstanding_Debt"]
    .str.replace("_", "", regex=False)
)

In [49]:
df["Outstanding_Debt"] = pd.to_numeric(
    df["Outstanding_Debt"],
    errors="coerce"
)

In [50]:
df["Amount_invested_monthly"].unique()

array(['80.41529543900253', '118.28022162236736', '81.699521264648', ...,
       '24.02847744864441', '251.67258219721603', '167.1638651610451'],
      shape=(91050,), dtype=object)

In [51]:
df["Amount_invested_monthly"] = pd.to_numeric(
    df["Amount_invested_monthly"],
    errors="coerce"
)

In [52]:
df["Monthly_Balance"].unique()

array(['312.49408867943663', '284.62916249607184', '331.2098628537912',
       ..., 516.8090832742814, 319.1649785257098, 393.6736955618808],
      shape=(98793,), dtype=object)

In [53]:
df["Monthly_Balance"] = pd.to_numeric(
    df["Monthly_Balance"],
    errors="coerce"
)

In [54]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 28 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   ID                        100000 non-null  object 
 1   Customer_ID               100000 non-null  object 
 2   Month                     100000 non-null  object 
 3   Name                      90015 non-null   object 
 4   Age                       100000 non-null  int64  
 5   SSN                       94428 non-null   object 
 6   Occupation                92938 non-null   object 
 7   Annual_Income             100000 non-null  float64
 8   Monthly_Inhand_Salary     84998 non-null   float64
 9   Num_Bank_Accounts         100000 non-null  int64  
 10  Num_Credit_Card           100000 non-null  int64  
 11  Interest_Rate             100000 non-null  int64  
 12  Num_of_Loan               100000 non-null  int64  
 13  Type_of_Loan              88592 non-null   ob

A conversão dos tipos de dados foi concluída com sucesso, foram corrigidas as inconsistências de tipo identificadas e permitindo que as variáveis numéricas sejam utilizadas corretamente nas próximas etapas do projeto.
Como etapa de validação, foi utilizado o método 'df.info()' para confirmar os tipos de dados após as conversões realizadas e verificar a ausência de inconsistências relacionadas a tipos mistos, identificados anteriormente.

5.4. Tratamento de valores ausentes

Após a identificação dos valores ausentes, foi definida uma estratégia de tratamento de acordo com o tipo de cada variável.
Para as variáveis numéricas, os valores ausentes foram preenchidos utilizando a mediana da respectiva coluna, por ser uma medida menos sensível á influência de valores extremos.
Para as variáveis categóricas, foi utilizada a categoria 'Unknown' quando a ausência representava falta de informação, presenvando a consistência dos dados para as etapas posteriores de análise e modelagem.

In [55]:
df.isnull().sum()

ID                              0
Customer_ID                     0
Month                           0
Name                         9985
Age                             0
SSN                          5572
Occupation                   7062
Annual_Income                   0
Monthly_Inhand_Salary       15002
Num_Bank_Accounts               0
Num_Credit_Card                 0
Interest_Rate                   0
Num_of_Loan                     0
Type_of_Loan                11408
Delay_from_due_date             0
Num_of_Delayed_Payment       7002
Changed_Credit_Limit         2091
Num_Credit_Inquiries         1965
Credit_Mix                  20195
Outstanding_Debt                0
Credit_Utilization_Ratio        0
Credit_History_Age           9030
Payment_of_Min_Amount       12007
Total_EMI_per_month             0
Amount_invested_monthly      8784
Payment_Behaviour            7600
Monthly_Balance              1209
Credit_Score                    0
dtype: int64

In [56]:
(df.isnull().sum() / len(df) * 100).sort_values(ascending=False)

Credit_Mix                  20.195
Monthly_Inhand_Salary       15.002
Payment_of_Min_Amount       12.007
Type_of_Loan                11.408
Name                         9.985
Credit_History_Age           9.030
Amount_invested_monthly      8.784
Payment_Behaviour            7.600
Occupation                   7.062
Num_of_Delayed_Payment       7.002
SSN                          5.572
Changed_Credit_Limit         2.091
Num_Credit_Inquiries         1.965
Monthly_Balance              1.209
Num_Bank_Accounts            0.000
Annual_Income                0.000
Age                          0.000
ID                           0.000
Customer_ID                  0.000
Month                        0.000
Interest_Rate                0.000
Num_Credit_Card              0.000
Num_of_Loan                  0.000
Delay_from_due_date          0.000
Outstanding_Debt             0.000
Credit_Utilization_Ratio     0.000
Total_EMI_per_month          0.000
Credit_Score                 0.000
dtype: float64

In [57]:
df["Credit_Mix"] = df["Credit_Mix"].fillna("Unknown")

In [58]:
df["Payment_of_Min_Amount"] = df["Payment_of_Min_Amount"].fillna("Unknown")

In [59]:
df["Type_of_Loan"] = df["Type_of_Loan"].fillna("No Loan")

In [60]:
df["Occupation"] = df["Occupation"].fillna("Unknown")

In [61]:
df["Payment_Behaviour"] = df["Payment_Behaviour"].fillna("Unknown")

In [62]:
df["Monthly_Inhand_Salary"] = df["Monthly_Inhand_Salary"].fillna(
    df["Monthly_Inhand_Salary"].median()
)

In [63]:
df["Amount_invested_monthly"] = df["Amount_invested_monthly"].fillna(
    df["Amount_invested_monthly"].median()
)

In [64]:
df["Num_of_Delayed_Payment"] = df["Num_of_Delayed_Payment"].fillna(
    df["Num_of_Delayed_Payment"].median()
)

In [65]:
df["Changed_Credit_Limit"] = df["Changed_Credit_Limit"].fillna(
    df["Changed_Credit_Limit"].median()
)

In [66]:
df["Num_Credit_Inquiries"] = df["Num_Credit_Inquiries"].fillna(
    df["Num_Credit_Inquiries"].median()
)

In [67]:
df["Monthly_Balance"] = df["Monthly_Balance"].fillna(
    df["Monthly_Balance"].median()
)

In [68]:
df.groupby("Customer_ID")["Name"].nunique(dropna=True)

Customer_ID
CUS_0x1000    1
CUS_0x1009    1
CUS_0x100b    1
CUS_0x1011    1
CUS_0x1013    1
             ..
CUS_0xff3     1
CUS_0xff4     1
CUS_0xff6     1
CUS_0xffc     1
CUS_0xffd     1
Name: Name, Length: 12500, dtype: int64

In [69]:
(df.groupby("Customer_ID")["Name"]
    .nunique(dropna=True) > 1).sum()

np.int64(0)

In [70]:
df["Name"] = (
    df.groupby("Customer_ID")["Name"]
        .transform(lambda x: x.ffill().bfill())
)

In [71]:
(df.groupby("Customer_ID")["SSN"]
    .nunique(dropna=True) > 1).sum()

np.int64(0)

In [72]:
df["SSN"] = (
    df.groupby("Customer_ID")["SSN"]
        .transform(lambda x: x.ffill().bfill())
)

In [73]:
df.isnull().sum().sort_values(ascending=False)

Credit_History_Age          9030
ID                             0
Month                          0
Customer_ID                    0
Age                            0
SSN                            0
Occupation                     0
Annual_Income                  0
Monthly_Inhand_Salary          0
Num_Bank_Accounts              0
Num_Credit_Card                0
Name                           0
Interest_Rate                  0
Num_of_Loan                    0
Delay_from_due_date            0
Type_of_Loan                   0
Changed_Credit_Limit           0
Num_Credit_Inquiries           0
Credit_Mix                     0
Num_of_Delayed_Payment         0
Outstanding_Debt               0
Credit_Utilization_Ratio       0
Payment_of_Min_Amount          0
Total_EMI_per_month            0
Amount_invested_monthly        0
Payment_Behaviour              0
Monthly_Balance                0
Credit_Score                   0
dtype: int64

A validação realizada por meio de 'df.isnull().sum()' confirmou que os valores ausentes das variáveis numéricas e categóricas tratadas foram preenchidos.
As colunas 'Name' e 'SSN', os valores ausentes foram recuperados a partir de outros registros pertencentes ao mesmo 'Customer_ID', preservando as informações originais sem a necessidade de imputação por métodos estatísticos.
A coluna 'Credit_History_Age' permaneceu com valores ausentes, pois armazena o tempo de histórico de crédito em formato textual. Antes da imputação, essa variável será transformada para uma representação numérica adequada, permitindo que o tratamento dos valores ausentes seja realizado de forma consistente.

5.5. Transformação da variável 'Credit_History_Age'

A variável 'Credit_History_Age' representa o tempo de histórico de crédito do cliente, porém seus valores estão armazenados em formato textual, o que impede sua utilização direta em análises estatísticas e modelos de Machine Learning.
Nesta etapa, a variável será convertida para uma representação numérica em meses, preservando a informação temporal e permitindo seu tratamento nas etapas posteriores.

In [74]:
df["Credit_History_Age"].head()

0    22 Years and 1 Months
1                      NaN
2    22 Years and 3 Months
3    22 Years and 4 Months
4    22 Years and 5 Months
Name: Credit_History_Age, dtype: object

In [75]:
import re

def converter_meses(valor):
    if pd.isna(valor):
        return np.nan

    resultado = re.match(r"(\d+)\s+Years?\s+and\s+(\d+)\s+Months?", valor)

    if resultado:
        anos = int(resultado.group(1))
        meses = int(resultado.group(2))
        return anos * 12 + meses

    return np.nan

In [76]:
df["Credit_History_Age"] = df["Credit_History_Age"].apply(converter_meses)

In [77]:
df["Credit_History_Age"].head()

0    265.0
1      NaN
2    267.0
3    268.0
4    269.0
Name: Credit_History_Age, dtype: float64

In [78]:
df["Credit_History_Age"] = df["Credit_History_Age"].fillna(
    df["Credit_History_Age"].median()
)

In [79]:
df["Credit_History_Age"].isnull().sum()

np.int64(0)

A transformação da variável foi concluída com sucesso, convertendo o tempo de histórico de crédito para uma representação numérica em meses. Após a conversão, os valores ausentes foram preenchidos utilizando a mediana da variável, permitindo sua utilização nas etapas de análise e modelagem sem perda de consistência dos dados.

5.6. Tratamento de valores inconsistentes

In [80]:
df.describe()

,Age,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Delay_from_due_date,Num_of_Delayed_Payment,Changed_Credit_Limit,Num_Credit_Inquiries,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Total_EMI_per_month,Amount_invested_monthly,Monthly_Balance
count,100000.000000,1.000000e+05,100000.000000,100000.000000,100000.00000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000
mean,110.649700,1.764157e+05,4029.084964,17.091280,22.47443,72.466040,3.009960,21.068780,29.738370,10.368345,27.326780,1426.220376,32.285173,220.997160,1403.118217,189.690637,401.755494
std,686.244717,1.429618e+06,2961.363540,117.404834,129.05741,466.422621,62.647879,14.860104,218.017612,6.719627,191.293766,1155.129026,5.116875,95.133546,8306.041270,191.527772,212.750017
min,-500.000000,7.005930e+03,303.645417,-1.000000,0.00000,1.000000,-100.000000,-5.000000,-3.000000,-6.490000,0.000000,0.230000,20.000000,1.000000,0.000000,0.000000,0.007760
25%,24.000000,1.945750e+04,1792.084167,3.000000,4.00000,8.000000,1.000000,10.000000,9.000000,5.420000,3.000000,566.072500,28.052567,154.000000,30.306660,77.017414,270.913865
50%,33.000000,3.757861e+04,3093.745000,6.000000,5.00000,13.000000,3.000000,18.000000,14.000000,9.400000,6.000000,1166.155000,32.305784,219.000000,69.249473,128.954538,336.731225
75%,42.000000,7.279092e+04,5371.525000,7.000000,7.00000,20.000000,5.000000,28.000000,18.000000,14.660000,9.000000,1945.962500,36.496663,292.000000,161.224249,220.039055,467.670597
max,8698.000000,2.419806e+07,15204.633333,1798.000000,1499.00000,5797.000000,1496.000000,67.000000,4397.000000,36.970000,2597.000000,4998.070000,50.000000,404.000000,82331.000000,1977.326102,1602.040519


In [81]:
df["Age"].value_counts().sort_index().head(20)

Age
-500     886
 14     1175
 15     1574
 16     1455
 17     1502
 18     2385
 19     2793
 20     2744
 21     2716
 22     2785
 23     2654
 24     2714
 25     2861
 26     2945
 27     2859
 28     2968
 29     2735
 30     2727
 31     2955
 32     2884
Name: count, dtype: int64

In [82]:
df["Age"].sort_values().tail(1890)

31288     102
56166     109
32555     111
5921      112
81398     115
         ... 
13372    8674
82335    8678
35557    8682
82739    8697
71732    8698
Name: Age, Length: 1890, dtype: int64

In [83]:
df.loc[(df["Age"] < 0) | (df["Age"] > 120), "Age"] = np.nan

df["Age"] = df["Age"].fillna(df["Age"].median())

In [84]:
df["Age"].value_counts().sort_index().tail(20)

Age
46.0     1621
47.0     1227
48.0     1385
49.0     1375
50.0     1273
51.0     1291
52.0     1356
53.0     1354
54.0     1311
55.0     1366
56.0      362
95.0        3
99.0        1
100.0       1
102.0       1
109.0       1
111.0       1
112.0       1
115.0       1
118.0       1
Name: count, dtype: int64

In [85]:
df["Annual_Income"].describe()

count    1.000000e+05
mean     1.764157e+05
std      1.429618e+06
min      7.005930e+03
25%      1.945750e+04
50%      3.757861e+04
75%      7.279092e+04
max      2.419806e+07
Name: Annual_Income, dtype: float64

In [86]:
df["Annual_Income"].value_counts().sort_index().head(20)

Annual_Income
7005.930    8
7006.035    8
7006.520    8
7011.685    8
7012.310    8
7019.435    8
7020.545    8
7021.910    8
7023.160    8
7039.745    8
7046.500    8
7055.840    8
7056.405    8
7059.455    8
7064.385    8
7077.870    8
7079.320    8
7080.700    8
7084.365    8
7085.390    7
Name: count, dtype: int64

In [87]:
df["Annual_Income"].value_counts().sort_index().tail(20)

Annual_Income
23713472.0    1
23743065.0    1
23775314.0    1
23784659.0    1
23822065.0    1
23834698.0    1
23871966.0    1
23884555.0    1
23912939.0    1
23917742.0    1
23942655.0    1
24008957.0    1
24065688.0    1
24096975.0    1
24105151.0    1
24105369.0    1
24160009.0    1
24177153.0    1
24188807.0    1
24198062.0    1
Name: count, dtype: int64

In [88]:
Q1 = df["Annual_Income"].quantile(0.25)
Q3 = df["Annual_Income"].quantile(0.75)

IQR = Q3 - Q1

limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

print(limite_inferior)
print(limite_superior)

-60542.630000000005
152791.05


In [89]:
(
    (df["Annual_Income"] < limite_inferior) |
    (df["Annual_Income"] > limite_superior)
).sum()

np.int64(2783)

In [ ]:
# df.loc[df["Annual_Income"] > limite_superior, "Annual_Income"] = np.nan

# df["Annual_Income"] = df["Annual_Income"].fillna(
#     df["Annual_Income"].median()
# )

In [98]:
df["Monthly_Inhand_Salary"].describe()

count    100000.000000
mean       4029.084964
std        2961.363540
min         303.645417
25%        1792.084167
50%        3093.745000
75%        5371.525000
max       15204.633333
Name: Monthly_Inhand_Salary, dtype: float64

In [99]:
df["Monthly_Inhand_Salary"].value_counts().sort_index().head(20)

Monthly_Inhand_Salary
303.645417    8
319.556250    7
332.128333    7
332.431250    6
333.596667    6
355.208333    8
357.255833    7
358.058333    6
361.603333    6
368.374167    7
373.071667    7
378.993333    7
379.390833    6
379.602917    6
380.649167    6
382.701667    7
391.053333    8
391.291250    8
391.890000    6
393.159167    6
Name: count, dtype: int64

In [100]:
df["Monthly_Inhand_Salary"].value_counts().sort_index().tail(20)

Monthly_Inhand_Salary
14855.556667    7
14855.930000    6
14856.483333    5
14862.283333    5
14866.446667    8
14867.813333    6
14880.383333    7
14929.540000    5
14958.336667    3
14960.250000    8
14978.336667    7
15038.316667    5
15066.783333    7
15090.076667    7
15091.086667    5
15101.940000    8
15115.190000    7
15136.696667    7
15167.180000    8
15204.633333    7
Name: count, dtype: int64

In [101]:
Q1 = df["Monthly_Inhand_Salary"].quantile(0.25)
Q3 = df["Monthly_Inhand_Salary"].quantile(0.75)

IQR = Q3 - Q1

limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

print(limite_inferior)
print(limite_superior)

-3577.0770833333363
10740.686250000004


In [102]:
(
    (df["Monthly_Inhand_Salary"] < limite_inferior) |
    (df["Monthly_Inhand_Salary"] > limite_superior)
).sum()

np.int64(4365)

In [ ]:
# df.loc[df["Monthly_Inhand_Salary"] > limite_superior, "Monthly_Inhand_Salary"] = np.nan

# df["Monthly_Inhand_Salary"] = df["Monthly_Inhand_Salary"].fillna(
#     df["Monthly_Inhand_Salary"].median()
# )

In [91]:
df["Num_Credit_Card"].describe()

count    100000.00000
mean         22.47443
std         129.05741
min           0.00000
25%           4.00000
50%           5.00000
75%           7.00000
max        1499.00000
Name: Num_Credit_Card, dtype: float64

In [92]:
df["Num_Credit_Card"].value_counts().sort_index().head(20)

Num_Credit_Card
0        13
1      2132
2      2149
3     13277
4     14030
5     18459
6     16559
7     16615
8      4956
9      4643
10     4860
11       36
15        3
16        2
17        1
18        1
20        1
21        1
22        2
24        1
Name: count, dtype: int64

In [93]:
df["Num_Credit_Card"].value_counts().sort_index().tail(20)

Num_Credit_Card
1475    4
1477    3
1478    1
1479    3
1480    2
1483    1
1484    2
1485    1
1486    2
1487    1
1489    1
1490    2
1492    2
1493    2
1494    1
1495    1
1496    2
1497    3
1498    3
1499    2
Name: count, dtype: int64

In [94]:
df["Num_Credit_Card"].value_counts().sort_index().loc[25:200]

Num_Credit_Card
25     5
26     1
27     3
28     4
29     2
      ..
194    2
195    2
197    1
199    1
200    2
Name: count, Length: 139, dtype: int64

In [95]:
Q1 = df["Num_Credit_Card"].quantile(0.25)
Q3 = df["Num_Credit_Card"].quantile(0.75)

IQR = Q3 - Q1

limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

print(limite_inferior)
print(limite_superior)

-0.5
11.5


In [96]:
(
    (df["Num_Credit_Card"] < limite_inferior) |
    (df["Num_Credit_Card"] > limite_superior)
).sum()

np.int64(2271)

In [97]:
df.loc[df["Num_Credit_Card"] > 11, "Num_Credit_Card"] = np.nan

df["Num_Credit_Card"] = df["Num_Credit_Card"].fillna(
    df["Num_Credit_Card"].median()
)

seleção de variaveis?

6. Tratamento

7. EDA (Análise Exploratória dos Dados)

8. Preparação

9. Modelagem

10. Avaliação

11. Conclusão